# Preprocesamiento y features v1

Se construye un dataset supervisado desde `dataset_groundtruth.csv`. Se imputan valores numéricos con la mediana y categóricos con la moda del train. El texto faltante se representa como una cadena vacía. Las métricas de engagement reciben `log(1+x)` y se crean likes/vistas y comentarios/vistas. Los ratios quedan indefinidos cuando faltan conteos o las vistas son cero y luego se imputan con la mediana de train.

El primer fold de `StratifiedGroupKFold` (5 folds, semilla 42) evita que un canal aparezca en train y test. `split` es metadato para posteriores experimento, no entra en el modelo. También se excluyen predicción/etiquetado de origen, procedencia e identificadores. `channel_id` se usa únicamente para agrupar.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

notebook_dir = Path.cwd()
if not (notebook_dir / "preprocessing.py").exists():
    notebook_dir = notebook_dir / "data_processing"
if not (notebook_dir / "preprocessing.py").exists():
    raise FileNotFoundError("No se encontró data_processing/preprocessing.py")
sys.path.insert(0, str(notebook_dir.resolve()))

from preprocessing import DEFAULT_INPUT_PATH, DEFAULT_OUTPUT_PATH, build_features

print(f"Entrada: {DEFAULT_INPUT_PATH}")
print(f"Salida:  {DEFAULT_OUTPUT_PATH}")

## 1. Inspección inicial

Se valida el esquema requerido, unicidad de `video_id` y valores faltantes antes de transformar.

In [ ]:
import pandas as pd

raw = pd.read_csv(DEFAULT_INPUT_PATH, keep_default_na=False)
print(f"Filas: {len(raw):,} | Columnas: {len(raw.columns)}")
missing_counts = raw.apply(lambda column: column.astype("string").str.strip().eq("").sum())
display(missing_counts.loc[lambda values: values.gt(0)].rename("faltantes").to_frame())
display(raw["classification_label"].value_counts().rename("videos").to_frame())
assert raw["video_id"].is_unique

## 2. Ingeniería de características, imputación y exportación

La función central conserva título, descripción y tags; codifica categorías como texto; mantiene duración en segundos; transforma vistas, likes, dislikes y comentarios; y calcula los dos ratios solicitados. Los campos `prediction`, `is_ground_truth`, `source_dataset`, `video_id`, `channel_id`, `channel_title`, `published_at`, `duration_iso` y `favorite_count` no se exportan como features. `favorite_count` es constante (cero) en el dataset inspeccionado.

In [ ]:
features, audit = build_features()
display(pd.Series(audit["missing_before"], name="faltantes antes de imputar").to_frame())
display(pd.Series(audit["numeric_imputation"], name="mediana de train").to_frame())
display(pd.Series(audit["categorical_imputation"], name="moda de train").to_frame())
display(pd.Series(audit["split_counts"], name="filas").to_frame())
print(f"Dataset guardado: {audit['output_path']}")
display(features.head())

## 3. Verificaciones del artefacto

El CSV `dataset_features_v1.csv` incluye la etiqueta objetivo y una columna auxiliar `split`; no incluye IDs, columnas de leakage ni columnas con faltantes. Para una validación cruzada posterior, las imputaciones deben volver a ajustarse dentro de cada fold de entrenamiento y nunca reutilizar estadísticas calculadas fuera de ese fold.

In [ ]:
exported = pd.read_csv(DEFAULT_OUTPUT_PATH, keep_default_na=False)
for excluded in ("video_id", "channel_id", "channel_title", "prediction", "is_ground_truth", "source_dataset", "published_at", "duration_iso", "favorite_count"):
    assert excluded not in exported.columns
assert "classification_label" in exported.columns and "split" in exported.columns
assert not exported.isna().any().any()
assert set(exported["split"]) == {"train", "test"}
print(f"OK: {len(exported):,} filas; {len(exported.columns) - 2} features; sin nulos ni columnas excluidas.")